# Hierarchical Cell Type Classification Training

End-to-end attention pooling (ScipherModel = AttentionPooling + WideNN) with MarginalizationLoss on blood cell hierarchy (CL:0000988).

**Runs on cluster** (needs SOMA database + ESM-2 embeddings).

## 1. Setup & Data Loading

In [ ]:
import sys
import logging
import pickle
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

# Find project root by searching upward for pyproject.toml
_cwd = Path(".").resolve()
PROJECT_ROOT = _cwd
for _parent in [_cwd] + list(_cwd.parents):
    if (_parent / "pyproject.toml").exists():
        PROJECT_ROOT = _parent
        break
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"
print(f"Project root: {PROJECT_ROOT}")

from scipher.hierarchy import (
    load_prebuilt_hierarchy, MarginalizationLoss, WideNN,
)
from scipher.embedders.attention import AttentionPooling, CellDataset

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

# --- Config ---
DATE = "2026-01-29"
ROOT_CL_ID = "CL:0000988"  # blood cells
SOMA_URI = "/scratch/sigbio_project_root/sigbio_project25/jingqiao/mccell-single/soma_db_homo_sapiens"
N_CELLS = 200_000
MIN_CELL_COUNT = 50
BATCH_SIZE = 256  # per GPU
LR = 1e-4
LEAF_WEIGHT = 7.0
GRAD_CLIP = 1.0
EPOCHS = 10
SEED = 42
NUM_WORKERS = 8

# Checkpoint directory: date_CLnumber
cl_folder = ROOT_CL_ID.replace(":", "")
checkpoint_dir = DATA_DIR / "checkpoints" / f"{DATE}_{cl_folder}"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()

# DataParallel splits the DataLoader batch across GPUs, so we scale up
# the DataLoader batch size so each GPU gets BATCH_SIZE samples
effective_batch_size = BATCH_SIZE * max(num_gpus, 1)

print(f"Device: {device}")
print(f"Checkpoints: {checkpoint_dir}")
if torch.cuda.is_available():
    print(f"GPUs: {num_gpus}x {torch.cuda.get_device_name(0)}")
    print(f"Batch size: {BATCH_SIZE}/GPU x {num_gpus} GPUs = {effective_batch_size} effective")
print(f"DataLoader workers: {NUM_WORKERS}")

In [ ]:
(
    mapping_dict, leaf_values, internal_values,
    marginalization_df, parent_child_df, exclusion_df,
) = load_prebuilt_hierarchy(DATE, ROOT_CL_ID)

all_cell_values = list(leaf_values) + list(internal_values)
print(f"Leaf types: {len(leaf_values)}, Internal: {len(internal_values)}, Total: {len(all_cell_values)}")

In [ ]:
EMB_PATH = DATA_DIR / "embeddings" / "gene_to_embedding.pkl"
with open(EMB_PATH, "rb") as f:
    gene_to_embedding = pickle.load(f)

embed_dim = next(iter(gene_to_embedding.values())).shape[0]
print(f"Gene embeddings: {len(gene_to_embedding):,} genes x {embed_dim}-dim")

In [ ]:
BIOMART_FILE = DATA_DIR / "raw" / "mart_export.txt"
df_biomart = pd.read_csv(BIOMART_FILE)
df_pc = df_biomart[df_biomart["Gene type"] == "protein_coding"].dropna(
    subset=["Gene stable ID", "Gene name"]
)
gene_list = df_pc["Gene stable ID"].unique().tolist()
print(f"Protein-coding Ensembl IDs: {len(gene_list):,}")

In [ ]:
import tiledbsoma as soma

experiment = soma.open(SOMA_URI, mode="r")

# Step 1: Read obs metadata
obs_df = (
    experiment.obs.read(
        value_filter='assay == "10x 3\' v3" and is_primary_data == True',
        column_names=["soma_joinid", "cell_type_ontology_term_id", "cell_type"],
    )
    .concat()
    .to_pandas()
)
print(f"Total 10x v3 primary cells: {len(obs_df):,}")

# Filter to hierarchy cell types
obs_df = obs_df[obs_df["cell_type_ontology_term_id"].isin(all_cell_values)].copy()
print(f"Blood cells in hierarchy: {len(obs_df):,}")

# Drop rare cell types
type_counts = obs_df["cell_type_ontology_term_id"].value_counts()
keep_types = type_counts[type_counts >= MIN_CELL_COUNT].index
obs_df = obs_df[obs_df["cell_type_ontology_term_id"].isin(keep_types)].copy()
print(
    f"After dropping < {MIN_CELL_COUNT} cells: {len(obs_df):,} cells, "
    f"{obs_df['cell_type_ontology_term_id'].nunique()} types"
)

# Subsample
if len(obs_df) > N_CELLS:
    obs_df = obs_df.sample(n=N_CELLS, random_state=SEED)
    print(f"Subsampled to {len(obs_df):,} cells")

# Step 2: Load expression for selected cells
var_value_filter = f"feature_id in {gene_list}"
joinids = obs_df["soma_joinid"].tolist()

with experiment.axis_query(
    measurement_name="RNA",
    obs_query=soma.AxisQuery(coords=(joinids,)),
    var_query=soma.AxisQuery(value_filter=var_value_filter),
) as query:
    adata = query.to_anndata(
        X_name="raw",
        column_names={
            "obs": ["cell_type_ontology_term_id", "cell_type"],
            "var": ["feature_id", "feature_name"],
        },
    )

adata.var_names = adata.var["feature_name"].values

print(f"\nLoaded AnnData: {adata.shape[0]:,} cells x {adata.shape[1]:,} genes")
print(f"Cell types: {adata.obs['cell_type_ontology_term_id'].nunique()}")
print(f"\nDistribution (top 15):")
print(adata.obs["cell_type"].value_counts().head(15).to_string())

In [ ]:
# Deduplicate gene symbols: keep first occurrence
seen = set()
dup_mask = []
for name in adata.var_names:
    if name in seen:
        dup_mask.append(False)
    else:
        seen.add(name)
        dup_mask.append(True)

n_dups = sum(1 for x in dup_mask if not x)
if n_dups > 0:
    adata = adata[:, dup_mask].copy()
adata.var_names_make_unique()

n_mapped = sum(1 for name in adata.var_names if name in gene_to_embedding)
print(f"Genes with embeddings: {n_mapped:,}/{adata.n_vars:,} ({100*n_mapped/adata.n_vars:.1f}%)")
print(f"Duplicates removed: {n_dups:,}")

In [ ]:
# Integer labels via mapping_dict
labels = np.array([mapping_dict[ct] for ct in adata.obs["cell_type_ontology_term_id"]])

# Reverse mappings for evaluation
idx_to_cl = {v: k for k, v in mapping_dict.items()}
cl_to_name = (
    adata.obs.drop_duplicates("cell_type_ontology_term_id")
    .set_index("cell_type_ontology_term_id")["cell_type"]
    .to_dict()
)
leaf_idx_set = {mapping_dict[cl] for cl in leaf_values}

# Stratified train/val split
train_idx, val_idx = train_test_split(
    np.arange(len(labels)),
    test_size=0.2,
    random_state=SEED,
    stratify=adata.obs["cell_type_ontology_term_id"],
)

train_adata = adata[train_idx].copy()
val_adata = adata[val_idx].copy()
train_labels = labels[train_idx]
val_labels = labels[val_idx]

n_train_leaf = sum(1 for l in train_labels if l in leaf_idx_set)
n_val_leaf = sum(1 for l in val_labels if l in leaf_idx_set)
print(f"Train: {len(train_labels):,} cells ({n_train_leaf:,} leaf-labeled)")
print(f"Val:   {len(val_labels):,} cells ({n_val_leaf:,} leaf-labeled)")

## 2. Model & Training

In [ ]:
class ScipherModel(nn.Module):
    def __init__(self, embed_dim, num_leaves, hidden_dim=256):
        super().__init__()
        self.embedder = AttentionPooling(embed_dim=embed_dim, hidden_dim=hidden_dim)
        self.classifier = WideNN(input_dim=embed_dim, output_dim=num_leaves)

    def forward(self, gene_embeddings, expression, mask):
        cell_embedding, attn_weights = self.embedder(gene_embeddings, expression, mask)
        logits = self.classifier(cell_embedding)
        return logits, cell_embedding, attn_weights

In [ ]:
train_dataset = CellDataset([train_adata], train_labels, gene_to_embedding)
val_dataset = CellDataset([val_adata], val_labels, gene_to_embedding)

train_loader = DataLoader(
    train_dataset, batch_size=effective_batch_size, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=effective_batch_size, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
model = ScipherModel(
    embed_dim=embed_dim, num_leaves=len(leaf_values), hidden_dim=256,
).to(device)
if num_gpus > 1:
    model = nn.DataParallel(model)
    print(f"Using DataParallel across {num_gpus} GPUs")

loss_fn = MarginalizationLoss(
    marginalization_df, parent_child_df, exclusion_df,
    leaf_values, internal_values, mapping_dict,
    leaf_weight=LEAF_WEIGHT, device=device,
)
optimizer = optim.Adam(model.parameters(), lr=LR)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ScipherModel: {n_params:,} trainable parameters")
attn_params = sum(p.numel() for n, p in model.named_parameters() if "embedder" in n)
cls_params = sum(p.numel() for n, p in model.named_parameters() if "classifier" in n)
print(f"  AttentionPooling: {attn_params:,}")
print(f"  WideNN classifier: {cls_params:,}")

In [ ]:
# Load gene embeddings once (shared across all batches)
gene_embs = train_dataset.get_gene_embeddings(device)

loss_history = {"total": [], "leaf": [], "parent": []}
epoch_times = []

print(f"Training: {EPOCHS} epochs, {len(train_loader)} batches/epoch")
print(f"Saving checkpoints to: {checkpoint_dir}")
print("-" * 70)

for epoch in range(EPOCHS):
    model.train()
    epoch_start = time.time()
    epoch_losses = []

    for i, batch in enumerate(train_loader):
        expression = batch["expression"].to(device)
        mask = batch["mask"].to(device)
        y_batch = batch["label"].to(device)

        # Expand gene embeddings to batch size (zero-copy view)
        batch_size = expression.size(0)
        embeddings = gene_embs.unsqueeze(0).expand(batch_size, -1, -1)

        optimizer.zero_grad()
        logits, _, _ = model(embeddings, expression, mask)
        total_loss, loss_leafs, loss_parents = loss_fn(logits, y_batch)
        total_loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        loss_history["total"].append(total_loss.item())
        loss_history["leaf"].append(loss_leafs.item())
        loss_history["parent"].append(loss_parents.item())
        epoch_losses.append(total_loss.item())

        if (i + 1) % 200 == 0:
            avg_recent = np.mean(epoch_losses[-200:])
            print(
                f"  [Epoch {epoch+1}/{EPOCHS}, Batch {i+1}/{len(train_loader)}] "
                f"Loss: {total_loss.item():.4f} (avg: {avg_recent:.4f})"
            )

    epoch_time = time.time() - epoch_start
    epoch_times.append(epoch_time)
    avg_loss = np.mean(epoch_losses)

    # Save checkpoint every epoch
    state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    ckpt = {
        "epoch": epoch + 1,
        "model_state_dict": state_dict,
        "optimizer_state_dict": optimizer.state_dict(),
        "loss_history": {k: list(v) for k, v in loss_history.items()},
        "epoch_times": list(epoch_times),
        "config": {
            "input_dim": embed_dim, "output_dim": len(leaf_values),
            "date": DATE, "root_cl_id": ROOT_CL_ID, "model_class": "ScipherModel",
            "lr": LR, "batch_size": BATCH_SIZE, "leaf_weight": LEAF_WEIGHT,
            "num_gpus": num_gpus,
        },
    }
    ckpt_path = checkpoint_dir / f"epoch{epoch+1:02d}.pt"
    torch.save(ckpt, ckpt_path)

    print(
        f"Epoch {epoch+1}/{EPOCHS} -- avg loss: {avg_loss:.4f}, "
        f"time: {epoch_time:.1f}s, saved: {ckpt_path.name}"
)

## 3. Evaluation

In [ ]:
gene_embs_val = val_dataset.get_gene_embeddings(device)
model.eval()
all_preds, all_labels = [], []
val_cell_embeddings = []

with torch.no_grad():
    for batch in val_loader:
        expression = batch["expression"].to(device)
        mask = batch["mask"].to(device)
        y_batch = batch["label"].to(device)

        batch_size = expression.size(0)
        embeddings = gene_embs_val.unsqueeze(0).expand(batch_size, -1, -1)

        logits, cell_emb, _ = model(embeddings, expression, mask)
        preds = torch.argmax(logits, dim=1)

        is_leaf = torch.tensor(
            [y.item() in leaf_idx_set for y in y_batch], device=device
        )
        if is_leaf.sum() > 0:
            all_preds.extend(preds[is_leaf].cpu().numpy())
            all_labels.extend(y_batch[is_leaf].cpu().numpy())

        val_cell_embeddings.append(cell_emb.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
val_embeddings = np.concatenate(val_cell_embeddings, axis=0)

acc = (all_preds == all_labels).mean()
macro_f1 = f1_score(all_labels, all_preds, average="macro")
weighted_f1 = f1_score(all_labels, all_preds, average="weighted")

print(f"=== Validation (leaf-labeled cells) ===")
print(f"  Leaf accuracy: {acc:.4f}")
print(f"  Macro F1:      {macro_f1:.4f}")
print(f"  Weighted F1:   {weighted_f1:.4f}")
print(f"  N samples:     {len(all_labels):,}")

# Per-class accuracy
rows = []
for idx in sorted(set(all_labels)):
    cl_id = idx_to_cl[idx]
    name = cl_to_name.get(cl_id, cl_id)
    mask_idx = all_labels == idx
    n = mask_idx.sum()
    class_acc = (all_preds[mask_idx] == all_labels[mask_idx]).mean()
    rows.append({"cell_type": name, "cl_id": cl_id, "n": n, "accuracy": class_acc})

df_eval = pd.DataFrame(rows).sort_values("accuracy")
print(f"\nPer-class accuracy:")
print(df_eval.to_string(index=False, float_format="{:.3f}".format))

In [ ]:
print("=" * 60)
print("Training Summary")
print("=" * 60)
print(f"  Leaf accuracy: {acc:.4f}")
print(f"  Macro F1:      {macro_f1:.4f}")
print(f"  Weighted F1:   {weighted_f1:.4f}")
print(f"  Epochs:        {EPOCHS}")
print(f"  Total time:    {sum(epoch_times):.1f}s")
print(f"  Checkpoints:   {checkpoint_dir}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

def smooth(values, window=50):
    kernel = np.ones(window) / window
    return np.convolve(values, kernel, mode="valid")

for ax, key, title in zip(
    axes, ["total", "leaf", "parent"],
    ["Total Loss", "Leaf Loss", "Parent Loss"],
):
    if len(loss_history[key]) > 50:
        ax.plot(smooth(loss_history[key]), alpha=0.8)
    else:
        ax.plot(loss_history[key], alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("Batch")
    ax.set_ylabel("Loss")

plt.tight_layout()
plt.savefig(DATA_DIR / "reports" / "training_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to data/reports/training_loss_curves.png")

In [ ]:
import scanpy as sc

sc.settings.set_figure_params(dpi=100, frameon=False)

# Build AnnData for UMAP visualization (val set only)
val_ct = val_adata.obs["cell_type"].values
adata_umap = sc.AnnData(
    X=np.zeros((len(val_ct), 1)),  # placeholder
    obs=pd.DataFrame({"cell_type": val_ct}),
)
adata_umap.obsm["X_attn"] = val_embeddings

sc.pp.neighbors(adata_umap, use_rep="X_attn")
sc.tl.umap(adata_umap)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
sc.pl.umap(
    adata_umap, color="cell_type", ax=ax, show=False,
    title="Attention-Pooled Cell Embeddings",
    legend_loc="on data", legend_fontsize=6, frameon=False,
)

plt.tight_layout()
plt.savefig(
    DATA_DIR / "reports" / "train_embedding_umap.png",
    dpi=150, bbox_inches="tight",
)
plt.show()
print("Saved to data/reports/train_embedding_umap.png")